# Notebook 02: Data Preparation and Integration

**Project:** Pharmacogenomics Machine Learning

**Author:** Sofia Muñoz

**Purpose:** This notebook prepares the raw ClinPGx datasets for machine learning by analyzing dataset relationships, determining merge strategies, cleaning the data, engineering features, and producing a final master dataset suitable for model development.

**Datasets:**
- Clinical Variants
- Variant Drug Annotations
- Variant Phenotype Annotations
- Variant Functional Assay Annotations

## Objectives

By the end of this notebook, I will:
- Identify relationships among all four datasets.
- Determine scientifically appropriate merge keys.
- Validate candidate merge keys.
- Develop a documented merge strategy.
- Clean and standardize each dataset.
- Integrate the datasets into a single master table.
- Engineer features for machine learning.
- Export a processed dataset for model development.

## Table of Contents

1. Import Libraries
2. Load Raw Datasets
3. Dataset Relationship Analysis
4. Candidate Merge Keys
5. Merge Strategy
6. Data Cleaning
7. Standardization
8. Missing Values
9. Duplicate Analysis
10. Dataset Integration
11. Feature Engineering
12. Final Quality Checks
13. Export Processed Dataset
14. Reflection

### 1. Import Libraries

In [22]:
# Import the pandas library for reading, manipulating, and analyzing tabular data.
import pandas as pd

# Import NumPy for numerical operations.
import numpy as np

# Import Matplotlib for creating graphs and visualizations.
import matplotlib.pyplot as plt

# Import Path from pathlib to build operating system-independent file paths.
from pathlib import Path

### 2. Load Raw Datasets

In [23]:
# Load the Clinical Variants dataset.
clinical_raw = pd.read_csv(
    "../data/raw/clinicalVariants.tsv",
    sep="\t"
)
# Load the Variant Drug Annotations dataset.
drug_raw = pd.read_csv(
    "../data/raw/var_drug_ann.tsv",
    sep="\t"
)
# Load the Variant Phenotype Annotations dataset.
phenotype_raw = pd.read_csv(
    "../data/raw/var_pheno_ann.tsv",
    sep="\t"
)
# Load the Variant Functional Assays Annotations dataset.
functional_raw = pd.read_csv(
    "../data/raw/var_fa_ann.tsv",
    sep="\t"
)

In [24]:
# Verify the files loaded
print("Clinical Variants:", clinical_raw.shape)
print("Drug Annotations:", drug_raw.shape)
print("Phenotype Annotations:", phenotype_raw.shape)
print("Functional Assays:", functional_raw.shape)

Clinical Variants: (5190, 6)
Drug Annotations: (12975, 22)
Phenotype Annotations: (14490, 25)
Functional Assays: (2153, 23)


In [25]:
# Create working copies of each dataset.
# All preprocessing will be performed on these copies, preserving the original raw datasets for reference.

clinical = clinical_raw.copy()
drug = drug_raw.copy()
phenotype = phenotype_raw.copy()
functional = functional_raw.copy()

In [26]:
# Verify the copies
print(clinical.shape == clinical_raw.shape)
print(drug.shape == drug_raw.shape)
print(phenotype.shape == phenotype_raw.shape)
print(functional.shape == functional_raw.shape)

True
True
True
True


### 3. Dataset Relationship Analysis

#### 3.1 Biological Roles of Each Dataset

| Dataset | What one row represents | Primary purpose |
| :--- | :--- | :--- |
| **Clinical Variants** | One curated clinical pharmacogenomic association describing how a specific genetic variant (or genotype/haplotype) influences a drug-related outcome (such as efficacy, toxicity, dosage, or metabolism) based on published clinical evidence. | Summarizes clinically actionable pharmacogenomic knowledge and recommendations for healthcare decision-making. |
| **Variant Drug Annotations** | One evidence record describing the relationship between a specific genetic variant and a particular drug, including the reported pharmacogenomic effect from an individual study or publication. Multiple rows may exist for the same variant because different drugs, studies, or evidence sources can be associated with it. | Provides detailed evidence linking genetic variants to drug response and serves as the primary source of variant–drug relationships. |
| **Variant Phenotype Annotations** | One evidence record describing how a specific genetic variant is associated with an observed phenotype (for example, altered metabolism, treatment response, or adverse drug reaction) reported in a publication. | Connects genetic variants with observed pharmacogenomic phenotypes that may explain differences in medication response. |
| **Variant Functional Assay Annotations** | One laboratory experimental result measuring the functional impact of a specific genetic variant on gene or protein activity. These data come from experimental assays rather than clinical observations. | Provides biological evidence about how variants affect molecular function, supporting interpretation of clinical and pharmacogenomic findings. |

#### 3.2 Candidate Shared Columns

| Column | Clinical | Drug | Phenotype | Functional |
| :--- | :---: | :---: | :---: | :---: |
| Variant Annotation ID | X | ✓ | ✓ | ✓ |
| Gene | ✓ | ✓ | ✓ | ✓ |
| Variant | ✓ | ✓ | ✓ | ✓ |
| Alleles | X | ✓ | ✓ | ✓ |
| Drug | ✓ | ✓ | ✓ | ✓ |
| PMID | X | ✓ | ✓ | ✓ |

#### 3.3 Candidate Merge Key Evaluation

##### Candidate Key 1: Variant Annotation ID

Variant Annotation ID is evaluated first because it appears in the three variant annotation datasets and is intended to uniquely identify a variant/drug annotation.

In [27]:
# Determine whether Variant Annotation ID exists in each dataset.

candidate_key = "Variant Annotation ID"

datasets = {
    "Clinical Variants": clinical,
    "Variant Drug Annotations": drug,
    "Variant Phenotype Annotations": phenotype,
    "Variant Functional Assay Annotations": functional
}
for name, df in datasets.items():
    print(f"{name}: {candidate_key in df.columns}")

Clinical Variants: False
Variant Drug Annotations: True
Variant Phenotype Annotations: True
Variant Functional Assay Annotations: True


*Observation*

Variant Annotation ID is present in the Variant Drug Annotations, Variant Phenotype Annotations, and Variant Functional Assay Annotations datasets but is absent from the Clinical Variants dataset.

This suggests that it may serve as the primary integration key among the variant annotation datasets but cannot directly link the Clinical Variants dataset.

In [28]:
for name, df in datasets.items():
    if candidate_key in df.columns:
        missing = df[candidate_key].isna().sum()
        print(f"{name}: {missing} missing values")

Variant Drug Annotations: 0 missing values
Variant Phenotype Annotations: 0 missing values
Variant Functional Assay Annotations: 0 missing values


In [29]:
for name, df in datasets.items():
    if candidate_key in df.columns:
        print(name)
        print("Unique:",
              df[candidate_key].is_unique)
        print()

Variant Drug Annotations
Unique: True

Variant Phenotype Annotations
Unique: True

Variant Functional Assay Annotations
Unique: True



In [30]:
summary = pd.DataFrame({
    "Question": [
        "Present in datasets?",
        "Unique?",
        "Missing values?",
        "Represents biology?",
        "Merge candidate?"
    ],
    "Answer": [
        "All but Clinical Variants",
        "Yes",
        "No",
        "Yes",
        "Yes"
    ]})
summary

,Question,Answer
0,Present in datasets?,All but Clinical Variants
1,Unique?,Yes
2,Missing values?,No
3,Represents biology?,Yes
4,Merge candidate?,Yes
